# BIDCell segmentation for decoded ISS transcripts

BIDCell requires DAPI plus an annotated single-cell reference. Pass an `.h5ad` file and its cell-type column; the wrapper automatically creates BIDCell's average-expression and positive/negative marker CSVs, generates the configuration, runs the pipeline in a separate environment, and converts the final connected TIFF to the standard sparse `.npz` mask.

Keep this notebook on the `ISS_postprocessing` kernel and create the isolated BIDCell runtime once from the repository root:

```bash
conda env create -f ISS_postprocessing/environments/bidcell.yml
```

The environment pins BIDCell 1.0.3, the compatible Cellpose 2/3 API, NumPy 1.26, OpenCV below 4.11, and a CUDA 11.8 PyTorch stack. BIDCell strongly benefits from an NVIDIA GPU. The wrapper launches it through `conda run`; do not change the notebook kernel to `bidcell`.

In [ ]:
from pathlib import Path
import ISS_postprocessing.segmentation as SEG

input_dir = Path('/path/to/regions')
region = 'R1'
transcripts_file = input_dir / region / 'decoding' / '2_decoded' / f'{region}_decoded.csv'
dapi_file = input_dir / region / 'preprocessing' / 'Cycle1' / '3_stitched' / 'Cycle1_ch4.tif'
reference_adata = Path('/path/to/annotated_reference.h5ad')
cell_type_col = 'cell_type'

In [ ]:
labels, labels_coo = SEG.bidcell_segmentation(
    transcripts=transcripts_file,
    image=dapi_file,
    region=region,
    input_dir=input_dir,
    reference_adata=reference_adata,
    cell_type_col=cell_type_col,
    # reference_layer='counts',  # if counts are stored in an AnnData layer
    # reference_use_raw=True,    # alternatively, use adata.raw
    pixel_size_um=0.325,
    target_pixel_size_um=1.0,
    bidcell_python=['conda', 'run', '-n', 'bidcell', 'python'],
    cpus=8,
    total_steps=4000,
)
labels.shape, int(labels.max()), labels_coo.nnz